<a href="https://colab.research.google.com/github/iandrukhiv-cell/addressbook-python/blob/main/%D0%90%D0%BD%D0%B4%D1%80%D1%83%D1%85%D1%96%D0%B2_%D0%86%D0%BB%D0%BE%D0%BD%D0%B0_%D0%92%D0%BE%D0%BB%D0%BE%D0%B4%D0%B8%D0%BC%D0%B8%D1%80%D1%96%D0%B2%D0%BD%D0%B0_%D0%9E%D1%81%D0%BD%D0%BE%D0%B2%D0%B8_%D0%B0%D0%BD%D0%B0%D0%BB%D1%96%D1%82%D0%B8%D0%BA%D0%B8_%D0%94%D0%974.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Збирання цитат з усіх сторінок сайту

In [18]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

base_url = "http://quotes.toscrape.com"
url = base_url

quotes_data = []

while url:
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    quotes = soup.find_all("div", class_="quote")

    for q in quotes:
        quote_text = q.find("span", class_="text").text
        author = q.find("small", class_="author").text
        tags = [tag.text for tag in q.find_all("a", class_="tag")]

        quotes_data.append({
            "quote": quote_text,
            "author": author,
            "tags": ",".join(tags)
        })

    next_btn = soup.find("li", class_="next")

    if next_btn:
        next_page = next_btn.find("a")["href"]
        url = base_url + next_page
    else:
        url = None

## Перевірка кількості зібраних цитат

In [19]:
len(quotes_data)

100

## Збереження цитат у файл quotes.csv

In [20]:
quotes_df = pd.DataFrame(quotes_data)
quotes_df.to_csv("quotes.csv", index=False)

quotes_df.head()

,quote,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"change,deep-thoughts,thinking,world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities,choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational,life,live,miracle,miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy,books,classic,humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself,inspirational"


## Збирання інформації про авторів

In [21]:
authors_data = []
visited_authors = set()

url = base_url

while url:
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    quotes = soup.find_all("div", class_="quote")

    for q in quotes:
        author = q.find("small", class_="author").text
        author_link = q.find("a")["href"]
        author_url = base_url + author_link

        if author not in visited_authors:
            visited_authors.add(author)

            author_response = requests.get(author_url)
            author_soup = BeautifulSoup(author_response.text, "html.parser")

            fullname = author_soup.find("h3", class_="author-title").text.strip()
            born_date = author_soup.find("span", class_="author-born-date").text.strip()
            born_location = author_soup.find("span", class_="author-born-location").text.strip()
            description = author_soup.find("div", class_="author-description").text.strip()

            authors_data.append({
                "fullname": fullname,
                "born_date": born_date,
                "born_location": born_location,
                "description": description
            })

    next_btn = soup.find("li", class_="next")

    if next_btn:
        next_page = next_btn.find("a")["href"]
        url = base_url + next_page
    else:
        url = None

## Перевірка кількості зібраних авторів

In [22]:
len(authors_data)

50

## Збереження авторів у файл authors.csv

In [23]:
authors_df = pd.DataFrame(authors_data)
authors_df.to_csv("authors.csv", index=False)

authors_df.head()

,fullname,born_date,born_location,description
0,Albert Einstein,"March 14, 1879","in Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germ..."
1,J.K. Rowling,"July 31, 1965","in Yate, South Gloucestershire, England, The U...",See also: Robert GalbraithAlthough she writes ...
2,Jane Austen,"December 16, 1775","in Steventon Rectory, Hampshire, The United Ki...",Jane Austen was an English novelist whose work...
3,Marilyn Monroe,"June 01, 1926",in The United States,Marilyn Monroe (born Norma Jeane Mortenson; Ju...
4,André Gide,"November 22, 1869","in Paris, France",André Paul Guillaume Gide was a French author ...


## Зчитування файлів quotes.csv та authors.csv

In [24]:
import pandas as pd

quotes = pd.read_csv("quotes.csv")
authors = pd.read_csv("authors.csv")

quotes.head(), authors.head()

(                                               quote           author  \
 0  “The world as we have created it is a process ...  Albert Einstein   
 1  “It is our choices, Harry, that show what we t...     J.K. Rowling   
 2  “There are only two ways to live your life. On...  Albert Einstein   
 3  “The person, be it gentleman or lady, who has ...      Jane Austen   
 4  “Imperfection is beauty, madness is genius and...   Marilyn Monroe   
 
                                        tags  
 0       change,deep-thoughts,thinking,world  
 1                         abilities,choices  
 2  inspirational,life,live,miracle,miracles  
 3             aliteracy,books,classic,humor  
 4                 be-yourself,inspirational  ,
           fullname          born_date  \
 0  Albert Einstein     March 14, 1879   
 1     J.K. Rowling      July 31, 1965   
 2      Jane Austen  December 16, 1775   
 3   Marilyn Monroe      June 01, 1926   
 4       André Gide  November 22, 1869   
 
                 

## Об’єднання цитат з інформацією про авторів

In [25]:
df = quotes.merge(authors, left_on="author", right_on="fullname")

df.head()

,quote,author,tags,fullname,born_date,born_location,description
0,“The world as we have created it is a process ...,Albert Einstein,"change,deep-thoughts,thinking,world",Albert Einstein,"March 14, 1879","in Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germ..."
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities,choices",J.K. Rowling,"July 31, 1965","in Yate, South Gloucestershire, England, The U...",See also: Robert GalbraithAlthough she writes ...
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational,life,live,miracle,miracles",Albert Einstein,"March 14, 1879","in Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germ..."
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy,books,classic,humor",Jane Austen,"December 16, 1775","in Steventon Rectory, Hampshire, The United Ki...",Jane Austen was an English novelist whose work...
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself,inspirational",Marilyn Monroe,"June 01, 1926",in The United States,Marilyn Monroe (born Norma Jeane Mortenson; Ju...


## Перетворення born_date у формат дати

In [26]:
df["born_date"] = pd.to_datetime(df["born_date"], errors="coerce")

df.dtypes

,0
quote,object
author,object
tags,object
fullname,object
born_date,datetime64[ns]
born_location,object
description,object


## Визначення ТОП-5 найпопулярніших тегів

In [27]:
from collections import Counter

df["tags"] = df["tags"].fillna("")

all_tags = df["tags"].str.split(",")

flat_tags = [tag for sublist in all_tags for tag in sublist if tag != ""]

top_5_tags = Counter(flat_tags).most_common(5)

top_5_tags

[('love', 14),
 ('inspirational', 13),
 ('life', 13),
 ('humor', 12),
 ('books', 11)]

## Пошук автора з найбільшою кількістю цитат

In [28]:
top_author = df["author"].value_counts().idxmax()
top_author

'Albert Einstein'

## Обчислення середньої довжини цитати для кожного автора

In [29]:
df["quote_length"] = df["quote"].str.len()

avg_quote_length = df.groupby("author")["quote_length"].mean().sort_values(ascending=False)
avg_quote_length

,quote_length
author,
Pablo Neruda,319.000000
Bob Marley,286.333333
J.D. Salinger,241.000000
Marilyn Monroe,240.857143
Elie Wiesel,224.000000
C.S. Lewis,184.400000
Jane Austen,171.800000
Ralph Waldo Emerson,168.500000
Charles Bukowski,161.500000


## Фільтрація авторів, які народилися у США

In [30]:
usa_authors = df[df["born_location"].str.contains("United States", na=False)]

usa_authors["author"].unique()

array(['Marilyn Monroe', 'Thomas A. Edison', 'Eleanor Roosevelt',
       'Steve Martin', 'Dr. Seuss', 'Mark Twain', 'Allen Saunders',
       'Ralph Waldo Emerson', 'Garrison Keillor', 'Jim Henson',
       'Charles M. Schulz', 'George R.R. Martin',
       'Martin Luther King Jr.', 'James Baldwin', 'Stephenie Meyer',
       'Ernest Hemingway', 'Helen Keller', 'Suzanne Collins',
       'J.D. Salinger', 'George Carlin', 'W.C. Fields', 'Jimi Hendrix',
       'E.E. Cummings', 'Harper Lee', "Madeleine L'Engle"], dtype=object)

In [31]:
print("TOP-5 тегів:", top_5_tags)
print("Топ автор:", top_author)
print("USA автори:", usa_authors["author"].unique())

TOP-5 тегів: [('love', 14), ('inspirational', 13), ('life', 13), ('humor', 12), ('books', 11)]
Топ автор: Albert Einstein
USA автори: ['Marilyn Monroe' 'Thomas A. Edison' 'Eleanor Roosevelt' 'Steve Martin'
 'Dr. Seuss' 'Mark Twain' 'Allen Saunders' 'Ralph Waldo Emerson'
 'Garrison Keillor' 'Jim Henson' 'Charles M. Schulz' 'George R.R. Martin'
 'Martin Luther King Jr.' 'James Baldwin' 'Stephenie Meyer'
 'Ernest Hemingway' 'Helen Keller' 'Suzanne Collins' 'J.D. Salinger'
 'George Carlin' 'W.C. Fields' 'Jimi Hendrix' 'E.E. Cummings' 'Harper Lee'
 "Madeleine L'Engle"]
